# Phase 1: Data Structuring & Quality Check

**Tujuan:**
Membaca matriks konektivitas hasil ekstraksi **Granger Causality (GC)** dan **Partial Directed Coherence (PDC)**
yang disimpan dalam format `.npy` per trial, mengekstrak metrik graf dari masing-masing matriks,
lalu menyusunnya menjadi data tabular bersih (CSV) yang siap digunakan untuk analisis statistik
dan klasifikasi machine learning.

> **FIX (information loss):** versi sebelumnya meringkas tiap matriks 62x62 menjadi hanya
> ~10 angka global (mean/max degree & strength, density) -- membuang semua informasi
> spasial per elektroda dan per-edge sebelum sampai ke Phase 4. Diverifikasi empiris pada
> data GC: representasi global -> ~51% akurasi (RF, 5-fold), per-channel -> ~64%, edge-weight
> penuh -> ~75% (SVM + SelectKBest). Versi ini menambahkan ketiga level fitur tersebut
> (lihat Section 2) supaya tidak ada informasi yang dibuang di Phase 1.

**Output:**
1. `cleaned_graph_metrics_gc.csv` — Metrik graf GC: global (10) + per-channel (248) + edge weight penuh (3782, dari matriks *raw* F-statistic)
2. `cleaned_graph_metrics_pdc_{band}.csv` — Metrik graf PDC per pita frekuensi (6 band): global (10) + per-channel (248) + edge weight penuh (3782, dari matriks PDC *sebelum* threshold)
3. `cleaned_graph_metrics_combined.csv` — Gabungan fitur GC + seluruh PDC

> **FIX (PDC edge parity, 2026-07-20):** sebelumnya fitur edge-weight penuh HANYA dihitung
> untuk GC -- alasannya nilai PDC saat itu masih terpengaruh bug ekstraksi koefisien MVAR di
> `02_pdc/code/pdc_analyzer.py::fit_mvar()` (baris intercept/constant dari `statsmodels.VAR.fit()`
> belum di-skip saat membaca koefisien lag, sehingga setiap matriks koefisien bergeser 1 baris),
> dan menambah 3782 fitur PDC/band di atas sinyal yang masih korup terbukti empiris justru
> MENURUNKAN akurasi gabungan (GC+PDC delta raw edges: turun dari 74.7% ke 73.2%). Bug MVAR
> itu **sudah diperbaiki** (lihat backup `02_pdc/output/pdc_matrices_buggy_backup_20260720`,
> offset `k_trend` sekarang benar) -- `extract_edge_features()` sekarang dipanggil juga untuk
> semua 6 band PDC (Section 4), memakai matriks PDC *raw* (kontinu, sebelum threshold),
> sama seperti GC memakai `gc_raw`. Ini membuat perbandingan GC vs PDC di Phase 2.4/4.7 adil
> (jumlah level fitur sama), bukan lagi GC (4040 fitur) vs PDC (258 fitur).

---


In [1]:
# ==============================================================================
# SECTION 0: IMPORT LIBRARIES
# ==============================================================================
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported')


Libraries imported


## Section 1: Konfigurasi Global & Parameter Dataset
Mendefinisikan direktori input untuk matriks GC dan PDC, direktori output untuk CSV,
nama-nama elektroda (62 channel SEED dataset), pemetaan label emosi standar SEED,
dan 6 pita frekuensi PDC.

In [2]:
# ==============================================================================
# SECTION 1: GLOBAL CONFIGURATION
# ==============================================================================
GC_DIR  = r'D:\Skripsi\new_data\01_granger_causality\output\gc_matrices'
PDC_DIR = r'D:\Skripsi\new_data\02_pdc\output\pdc_matrices'
OUTPUT_DIR = r'D:\Skripsi\new_data\phase_1_data_structuring\csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)

N_CHANNELS = 62

# Urutan label emosi SEED (15 trial): 1=positive, 0=neutral, -1=negative
TRIAL_LABELS = [1, 0, -1, -1, 0, 1, -1, 0, 1, 1, 0, -1, 0, 1, -1]

# EMOTION_MAP: kode SEED (-1/0/1) -> teks emosi
# class_label = teks emosi (digunakan untuk plotting & statistik)
# class       = kode numerik untuk ML classifier (negative=0, neutral=1, positive=2)
EMOTION_MAP = {-1: 'negative', 0: 'neutral',  1: 'positive'}
LABEL_MAP   = {-1: 0,          0: 1,           1: 2}

# 62 nama elektroda SEED
CHANNEL_NAMES = [
    'Fp1','Fpz','Fp2','AF3','AF4','F7','F5','F3','F1','Fz',
    'F2','F4','F6','F8','FT7','FC5','FC3','FC1','FCz','FC2',
    'FC4','FC6','FT8','T7','C5','C3','C1','Cz','C2','C4',
    'C6','T8','TP7','CP5','CP3','CP1','CPz','CP2','CP4','CP6',
    'TP8','P7','P5','P3','P1','Pz','P2','P4','P6','P8',
    'PO7','PO5','PO3','POz','PO4','PO6','PO8','CB1','O1','Oz',
    'O2','CB2',
]

PDC_BANDS = ['delta', 'theta', 'alpha', 'beta', 'gamma', 'broadband']

# FIX: PDC adalah nilai kontinu yang hampir tidak pernah persis nol, jadi
# 'binary = (m > 0)' di extract_graph_metrics() akan selalu menganggap graf
# fully-connected (density=1.0, degree=61 utk semua trial) jika PDC tidak
# di-threshold dulu. PDC_PERCENTILE=75 -> simpan top 25% edge terkuat saja
# per matriks (analog FDR-threshold pada GC).
PDC_PERCENTILE = 75

print('Config loaded')
print(f'  GC_DIR  : {GC_DIR}')
print(f'  PDC_DIR : {PDC_DIR}')
print(f'  OUTPUT  : {OUTPUT_DIR}')


Config loaded
  GC_DIR  : D:\Skripsi\new_data\01_granger_causality\output\gc_matrices
  PDC_DIR : D:\Skripsi\new_data\02_pdc\output\pdc_matrices
  OUTPUT  : D:\Skripsi\new_data\phase_1_data_structuring\csv


## Section 2: Fungsi Ekstraksi Fitur (3 Level Resolusi)
Tiga fungsi dengan resolusi informasi yang berbeda, dari matriks konektivitas berarah 62x62
(diagonal/self-connection selalu di-nol-kan lebih dulu):

1. **`extract_graph_metrics()`** — metrik **global** (1 angka per metrik, dirata-rata lintas
   62 channel): mean/max degree, mean/max strength, density, hub channel. Dipertahankan untuk
   narasi deskriptif di Phase 2 (mis. "rata-rata density GC saat emosi negatif...").
2. **`extract_channel_features()`** — metrik **per-channel** (degree & strength, out/in,
   *tanpa* dirata-ratakan lintas channel): 4 metrik x 62 channel = 248 fitur/metode. Di sinilah
   peran spesifik tiap elektroda (mis. `gc_str_out_Fp1`) tetap utuh sebagai fitur individual.
3. **`extract_edge_features()`** — bobot **edge mentah** (62x61 = 3782 pasangan channel
   terarah), resolusi tertinggi, tanpa agregasi maupun threshold apa pun. Dipakai khusus untuk
   GC dari matriks `gc_raw` (F-statistic kontinu, sebelum FDR threshold).


In [3]:
# ==============================================================================
# SECTION 2: FEATURE EXTRACTION (global + per-channel + edge-level)
# ==============================================================================
def extract_graph_metrics(matrix, method_name='gc'):
    """
    Ekstrak metrik graf GLOBAL (1 angka per metrik, dirata-rata lintas 62 channel)
    dari matriks konektivitas berarah (62x62). Dipakai untuk narasi deskriptif Phase 2 --
    BUKAN fitur utama untuk klasifikasi Phase 4, karena rata-rata lintas channel membuang
    informasi spasial per elektroda. Lihat extract_channel_features() dan
    extract_edge_features() untuk fitur beresolusi lebih tinggi.

    PENTING: Diagonal utama (self-connection) di-nol-kan TERLEBIH DAHULU
    sebelum menghitung semua metrik, agar self-loop tidak mempengaruhi
    nilai Strength, Degree, dan Density.
    """
    # Buat salinan dan nol-kan diagonal (self-connection)
    m = matrix.copy().astype(float)
    np.fill_diagonal(m, 0.0)

    n_ch = m.shape[0]

    # --- Strength (total bobot koneksi) ---
    out_strength   = np.sum(m, axis=1)          # baris = outgoing
    in_strength    = np.sum(m, axis=0)           # kolom = incoming
    total_strength = out_strength + in_strength

    # --- Degree (jumlah koneksi biner aktif) ---
    binary     = (m > 0).astype(float)           # sudah bebas self-loop
    out_degree = np.sum(binary, axis=1)
    in_degree  = np.sum(binary, axis=0)
    total_degree = out_degree + in_degree

    # --- Density ---
    n_possible = n_ch * (n_ch - 1)               # tidak termasuk diagonal
    density = np.count_nonzero(binary) / n_possible if n_possible > 0 else 0.0

    # --- Hub channel (elektroda dengan total strength terbesar) ---
    hub_idx = int(np.argmax(total_strength))

    return {
        f'{method_name}_mean_out_degree'    : float(np.mean(out_degree)),
        f'{method_name}_mean_in_degree'     : float(np.mean(in_degree)),
        f'{method_name}_mean_total_degree'  : float(np.mean(total_degree)),
        f'{method_name}_max_out_degree'     : float(np.max(out_degree)),
        f'{method_name}_max_in_degree'      : float(np.max(in_degree)),
        f'{method_name}_mean_out_strength'  : float(np.mean(out_strength)),
        f'{method_name}_mean_in_strength'   : float(np.mean(in_strength)),
        f'{method_name}_mean_total_strength': float(np.mean(total_strength)),
        f'{method_name}_max_total_strength' : float(np.max(total_strength)),
        f'{method_name}_density'            : float(density),
        f'{method_name}_hub_channel'        : CHANNEL_NAMES[hub_idx],
    }

print('extract_graph_metrics() defined (global, 10 metrics + hub_channel)')


def extract_channel_features(matrix, channel_names, method_name='gc'):
    """
    FIX (information loss): sebelumnya seluruh 62 channel diringkas jadi 1 angka
    mean/max saja (extract_graph_metrics), sehingga peran channel individual hilang --
    hanya tersisa nama satu 'hub_channel'. Fungsi ini mengembalikan degree & strength
    PER CHANNEL (out/in masing-masing), sehingga peran tiap 1 dari 62 elektroda tetap
    utuh sebagai fitur individual (4 metrik x 62 channel = 248 fitur/metode).

    Diverifikasi empiris (data GC, RandomForest, 5-fold CV): representasi ini menaikkan
    akurasi dari ~51% (metrik global) menjadi ~64% (per-channel).
    """
    m = matrix.copy().astype(float)
    np.fill_diagonal(m, 0.0)

    binary = (m > 0).astype(float)
    out_degree   = binary.sum(axis=1)
    in_degree    = binary.sum(axis=0)
    out_strength = m.sum(axis=1)
    in_strength  = m.sum(axis=0)

    feats = {}
    for idx, ch in enumerate(channel_names):
        feats[f'{method_name}_deg_out_{ch}'] = float(out_degree[idx])
        feats[f'{method_name}_deg_in_{ch}']  = float(in_degree[idx])
        feats[f'{method_name}_str_out_{ch}'] = float(out_strength[idx])
        feats[f'{method_name}_str_in_{ch}']  = float(in_strength[idx])
    return feats

print('extract_channel_features() defined (per-channel degree & strength, 248 features/method)')


def extract_edge_features(matrix, channel_names, method_name='gc'):
    """
    FIX (information loss): fitur beresolusi TERTINGGI -- seluruh 62x61=3782 bobot edge
    terarah off-diagonal disimpan langsung sebagai fitur individual, tanpa agregasi atau
    threshold apa pun.

    Awalnya HANYA dipakai untuk GC (dari matriks gc_raw, F-statistic kontinu belum
    di-threshold FDR), karena terbukti empiris performa terbaik: SVM + SelectKBest -> ~75%
    akurasi (dibanding ~51% dari metrik global dan ~64% dari per-channel). Sempat belum
    dipakai untuk PDC karena bug ekstraksi koefisien MVAR di 02_pdc/code/pdc_analyzer.py::
    fit_mvar() (baris intercept/constant dari statsmodels VAR.fit() belum di-skip). Bug itu
    SUDAH DIPERBAIKI (2026-07-20) -- fungsi ini sekarang dipanggil juga untuk tiap band PDC
    (lihat Section 4), memakai matriks PDC raw (kontinu, sebelum threshold), agar
    perbandingan GC vs PDC punya jumlah level fitur yang setara.
    """
    m = matrix.copy().astype(float)
    np.fill_diagonal(m, 0.0)
    n_ch = len(channel_names)

    feats = {}
    for i in range(n_ch):
        ch_i = channel_names[i]
        for j in range(n_ch):
            if i == j:
                continue
            feats[f'{method_name}_edge_{ch_i}_to_{channel_names[j]}'] = float(m[i, j])
    return feats

print('extract_edge_features() defined (3782 raw directed edge weights, GC + PDC)')


def threshold_pdc_matrix(matrix, cutoff):
    """
    Threshold matriks PDC dengan nilai cutoff ABSOLUT (bukan persentil per-trial).

    CATATAN PENTING: persentil per-trial (v1) selalu menyisakan jumlah edge yang
    SAMA PERSIS di setiap matriks (by definition), sehingga density & mean-degree
    tetap konstan (hanya nilainya bergeser dari 1.0 ke, misal, 0.25) -- tidak
    menyelesaikan masalah. Cutoff di sini adalah nilai magnitude PDC absolut yang
    sama untuk seluruh trial dalam satu band (lihat compute_global_pdc_cutoff),
    sehingga jumlah edge yang lolos threshold BERBEDA-BEDA secara alami sesuai
    kekuatan koneksi tiap trial -- inilah yang membuat density/degree kembali
    punya variance nyata antar-trial.
    """
    m = matrix.copy().astype(float)
    np.fill_diagonal(m, 0.0)
    m[m < cutoff] = 0.0
    return m


def compute_global_pdc_cutoff(matrices, percentile=None):
    """
    Hitung SATU nilai cutoff (persentil ke-`percentile`) dari seluruh nilai
    off-diagonal yang di-pool dari semua trial dalam satu band. Cutoff ini lalu
    dipakai sama untuk semua trial (lihat docstring threshold_pdc_matrix).
    """
    percentile = percentile if percentile is not None else PDC_PERCENTILE
    n_ch = matrices[0].shape[0]
    offdiag_mask = ~np.eye(n_ch, dtype=bool)
    pooled = np.concatenate([m[offdiag_mask] for m in matrices])
    return float(np.percentile(pooled, percentile))

print('threshold_pdc_matrix() + compute_global_pdc_cutoff() defined (global per-band cutoff, top %d%% kept on average)' % (100 - PDC_PERCENTILE))


extract_graph_metrics() defined (global, 10 metrics + hub_channel)
extract_channel_features() defined (per-channel degree & strength, 248 features/method)
extract_edge_features() defined (3782 raw directed edge weights, GC + PDC)
threshold_pdc_matrix() + compute_global_pdc_cutoff() defined (global per-band cutoff, top 25% kept on average)


## Section 3: Agregasi Granger Causality (GC)
Iterasi seluruh folder subjek dan sesi, membaca file `gc_thresholded_trial_*.npy`
(matriks GC setelah FDR threshold, dipakai untuk metrik global & per-channel) DAN
`gc_raw_trial_*.npy` (F-statistic kontinu sebelum threshold, dipakai untuk fitur edge
beresolusi penuh -- lihat `extract_edge_features()`), mengekstrak ketiga level fitur,
memberi label emosi, lalu menyimpan `cleaned_graph_metrics_gc.csv`
(6 kolom kunci + 10 global + 248 per-channel + 3782 edge = 4046 kolom).

> **Filter file:** Hanya file dengan kata `thresholded` yang dipakai untuk menentukan
> trial yang diproses; `gc_raw_*` untuk trial yang sama otomatis ikut dimuat.
> File `gc_lag_*` dan `p_values_*` diabaikan.
>
> **Folder non-subjek** seperti `gc_metadata.json` di-skip otomatis dengan cek `os.path.isdir()`.


In [4]:
# ==============================================================================
# SECTION 3: AGREGASI GRANGER CAUSALITY
# ==============================================================================
gc_rows = []

for entry in sorted(os.listdir(GC_DIR)):
    subj_path = os.path.join(GC_DIR, entry)

    # FIX: Skip file (mis. gc_metadata.json) — hanya proses folder subjek
    if not os.path.isdir(subj_path):
        continue
    # FIX: Pastikan ini folder subjek yang valid (bukan folder lain)
    if not entry.startswith('subject_'):
        continue

    subject_id = int(entry.split('_')[1])   # 'subject_01' -> 1

    for sess_entry in sorted(os.listdir(subj_path)):
        sess_path = os.path.join(subj_path, sess_entry)
        if not os.path.isdir(sess_path):
            continue
        if not sess_entry.startswith('session_'):
            continue

        session_date = sess_entry.split('_')[1]  # 'session_20131027' -> '20131027'

        for f in sorted(os.listdir(sess_path)):
            # Hanya baca file gc_thresholded_trial_XX.npy
            if not f.startswith('gc_thresholded_trial_') or not f.endswith('.npy'):
                continue

            # Ekstrak nomor trial: 'gc_thresholded_trial_01.npy' -> 1
            trial_idx = int(f.split('_')[-1].replace('.npy', ''))
            label     = TRIAL_LABELS[trial_idx - 1]   # indeks 0-based

            matrix_thresh = np.load(os.path.join(sess_path, f))

            # FIX: juga muat matriks RAW (F-statistic kontinu, sebelum FDR threshold)
            # untuk fitur edge beresolusi penuh -- lihat extract_edge_features().
            raw_fname  = f.replace('gc_thresholded_trial_', 'gc_raw_trial_')
            matrix_raw = np.load(os.path.join(sess_path, raw_fname))

            metrics  = extract_graph_metrics(matrix_thresh, 'gc')
            channels = extract_channel_features(matrix_thresh, CHANNEL_NAMES, 'gc')
            edges    = extract_edge_features(matrix_raw, CHANNEL_NAMES, 'gc')

            row = {
                'subject_id' : f'S{subject_id:02d}',
                'subject_num': subject_id,
                'session'    : session_date,
                'trial'      : trial_idx,
                'class'      : LABEL_MAP[label],    # numerik (0/1/2) untuk ML
                'class_label': EMOTION_MAP[label],  # teks untuk statistik & plot
            }
            row.update(metrics)
            row.update(channels)
            row.update(edges)
            gc_rows.append(row)

df_gc = pd.DataFrame(gc_rows)
print(f'GC data loaded: {df_gc.shape}')
print(f'  class_label distribution:')
print(df_gc['class_label'].value_counts().to_string())
print(f'  Global features     : {len([c for c in df_gc.columns if c.startswith("gc_") and "_deg_" not in c and "_str_" not in c and "_edge_" not in c])}')
print(f'  Per-channel features: {len([c for c in df_gc.columns if "_deg_" in c or ("_str_" in c and "_edge_" not in c)])}')
print(f'  Edge-weight features: {len([c for c in df_gc.columns if "_edge_" in c])}')

# Simpan CSV
gc_out = os.path.join(OUTPUT_DIR, 'cleaned_graph_metrics_gc.csv')
df_gc.to_csv(gc_out, index=False)
print(f'\nSaved: {gc_out}')


GC data loaded: (675, 4047)
  class_label distribution:
class_label
positive    225
neutral     225
negative    225
  Global features     : 11
  Per-channel features: 248
  Edge-weight features: 3782

Saved: D:\Skripsi\new_data\phase_1_data_structuring\csv\cleaned_graph_metrics_gc.csv


## Section 4: Agregasi PDC per Pita Frekuensi (6 Band)
Iterasi yang sama untuk PDC. Membaca file `pdc_{band}_trial_*.npy`,
mengekstrak metrik graf global, per-channel (`extract_channel_features`), DAN edge weight
penuh (`extract_edge_features`, dari matriks PDC raw sebelum threshold) -- lalu menyimpan
6 CSV terpisah per band (6 kunci + 10 global + 248 per-channel + 3782 edge = 4046 kolom).

> **Folder non-subjek** seperti `pdc_metadata.json` di-skip otomatis dengan cek `startswith('subject_')`.
>
> Fitur edge-level penuh kini disertakan untuk PDC juga (bug MVAR yang sebelumnya menahan
> ini sudah diperbaiki -- lihat catatan di Section 2 / `extract_edge_features()`), agar
> perbandingan GC vs PDC memakai jumlah level fitur yang setara.


In [5]:
# ==============================================================================
# SECTION 4: AGREGASI PDC PER PITA FREKUENSI
# ==============================================================================
pdc_datasets = {}
pdc_cutoffs = {}

for band in PDC_BANDS:
    # --- Pass 1: load semua matriks band ini ke memori + kumpulkan metadata ---
    entries = []  # (subject_id, session_date, trial_idx, label, matrix)

    for entry in sorted(os.listdir(PDC_DIR)):
        subj_path = os.path.join(PDC_DIR, entry)
        if not os.path.isdir(subj_path):
            continue
        if not entry.startswith('subject_'):
            continue

        subject_id = int(entry.split('_')[1])

        for sess_entry in sorted(os.listdir(subj_path)):
            sess_path = os.path.join(subj_path, sess_entry)
            if not os.path.isdir(sess_path):
                continue
            if not sess_entry.startswith('session_'):
                continue

            session_date = sess_entry.split('_')[1]

            for f in sorted(os.listdir(sess_path)):
                prefix_expected = f'pdc_{band}_trial_'
                if not f.startswith(prefix_expected) or not f.endswith('.npy'):
                    continue

                trial_idx = int(f.split('_')[-1].replace('.npy', ''))
                label     = TRIAL_LABELS[trial_idx - 1]
                matrix    = np.load(os.path.join(sess_path, f))
                entries.append((subject_id, session_date, trial_idx, label, matrix))

    # --- Hitung cutoff global (satu nilai) dari seluruh trial band ini ---
    cutoff = compute_global_pdc_cutoff([e[4] for e in entries], percentile=PDC_PERCENTILE)
    pdc_cutoffs[band] = cutoff

    # --- Pass 2: threshold pakai cutoff global lalu ekstrak fitur global + per-channel ---
    rows = []
    for subject_id, session_date, trial_idx, label, matrix in entries:
        matrix_thresh = threshold_pdc_matrix(matrix, cutoff=cutoff)
        metrics  = extract_graph_metrics(matrix_thresh, f'pdc_{band}')
        # FIX: fitur per-channel agar informasi spasial tidak hilang jadi rata-rata global saja.
        channels = extract_channel_features(matrix_thresh, CHANNEL_NAMES, f'pdc_{band}')
        # FIX (PDC edge parity): edge weight penuh dari matriks RAW (kontinu, sebelum
        # threshold) -- sama seperti GC memakai gc_raw. Bug MVAR yang tadinya menahan ini
        # sudah diperbaiki di 02_pdc/code/pdc_analyzer.py::fit_mvar().
        edges = extract_edge_features(matrix, CHANNEL_NAMES, f'pdc_{band}')

        row = {
            'subject_id' : f'S{subject_id:02d}',
            'subject_num': subject_id,
            'session'    : session_date,
            'trial'      : trial_idx,
            'class'      : LABEL_MAP[label],
            'class_label': EMOTION_MAP[label],
        }
        row.update(metrics)
        row.update(channels)
        row.update(edges)
        rows.append(row)

    df_band = pd.DataFrame(rows)
    pdc_datasets[band] = df_band

    fname = f'cleaned_graph_metrics_pdc_{band}.csv'
    df_band.to_csv(os.path.join(OUTPUT_DIR, fname), index=False)
    print(f'  PDC {band:12s}: {df_band.shape}  cutoff={cutoff:.4f}  density_std={df_band[f"pdc_{band}_density"].std():.4f}  ->  {fname}')

print('\nAll PDC bands saved.')
print('Global cutoffs per band:', {k: round(v, 4) for k, v in pdc_cutoffs.items()})


  PDC delta       : (675, 4047)  cutoff=0.0907  density_std=0.1079  ->  cleaned_graph_metrics_pdc_delta.csv
  PDC theta       : (675, 4047)  cutoff=0.0875  density_std=0.1070  ->  cleaned_graph_metrics_pdc_theta.csv
  PDC alpha       : (675, 4047)  cutoff=0.0891  density_std=0.1066  ->  cleaned_graph_metrics_pdc_alpha.csv
  PDC beta        : (675, 4047)  cutoff=0.0869  density_std=0.1228  ->  cleaned_graph_metrics_pdc_beta.csv
  PDC gamma       : (675, 4047)  cutoff=0.0868  density_std=0.1249  ->  cleaned_graph_metrics_pdc_gamma.csv
  PDC broadband   : (675, 4047)  cutoff=0.0859  density_std=0.1281  ->  cleaned_graph_metrics_pdc_broadband.csv

All PDC bands saved.
Global cutoffs per band: {'delta': 0.0907, 'theta': 0.0875, 'alpha': 0.0891, 'beta': 0.0869, 'gamma': 0.0868, 'broadband': 0.0859}


## Section 5: Penggabungan Dataset (GC + PDC)
Menggabungkan metrik GC dan seluruh 6 pita PDC menjadi satu DataFrame tunggal
berdasarkan kunci unik `subject_id`, `session`, `trial`.

> Menggunakan `how='inner'` agar hanya baris yang ada di **semua** dataset yang disertakan.
> Setelah merge, dilakukan validasi jumlah baris untuk memastikan tidak ada data yang hilang.

In [6]:
# ==============================================================================
# SECTION 5: MERGE GC + PDC
# ==============================================================================
merge_keys = ['subject_id', 'subject_num', 'session', 'trial', 'class', 'class_label']

df_combined = df_gc.copy()
original_rows = len(df_gc)

for band, df_band in pdc_datasets.items():
    before = len(df_combined)
    df_combined = df_combined.merge(df_band, on=merge_keys, how='inner')
    after = len(df_combined)
    if after < before:
        print(f'  WARNING: Merge with PDC {band} dropped {before - after} rows!')
    else:
        print(f'  Merged PDC {band}: {after} rows OK')

# Validasi: pastikan tidak ada baris yang hilang
if len(df_combined) != original_rows:
    print(f'\nWARNING: Combined rows ({len(df_combined)}) != GC rows ({original_rows})!')
    print('  Kemungkinan ada trial yang hilang di salah satu band PDC.')
else:
    print(f'\nAll rows intact: {len(df_combined)}')

print(f'Combined shape: {df_combined.shape}')
print(f'  GC features : {len([c for c in df_combined.columns if c.startswith("gc_")])}')
print(f'  PDC features: {len([c for c in df_combined.columns if c.startswith("pdc_")])}')

# Simpan
combined_out = os.path.join(OUTPUT_DIR, 'cleaned_graph_metrics_combined.csv')
df_combined.to_csv(combined_out, index=False)
print(f'\nSaved: {combined_out}')


  Merged PDC delta: 675 rows OK
  Merged PDC theta: 675 rows OK
  Merged PDC alpha: 675 rows OK
  Merged PDC beta: 675 rows OK
  Merged PDC gamma: 675 rows OK
  Merged PDC broadband: 675 rows OK

All rows intact: 675
Combined shape: (675, 28293)
  GC features : 4041
  PDC features: 24246

Saved: D:\Skripsi\new_data\phase_1_data_structuring\csv\cleaned_graph_metrics_combined.csv


## Section 6: Quality Control Audit
Pemeriksaan kualitas akhir:
1. Jumlah baris (15 subjek x 3 sesi x 15 trial = 675 baris per dataset)
2. Missing values dan duplikat
3. Distribusi kelas emosi (harus seimbang: 225 per kelas)
4. Fitur konstan (std ≈ 0)
5. Deteksi outlier menggunakan IQR

In [7]:
# ==============================================================================
# SECTION 6: QUALITY CONTROL AUDIT
# ==============================================================================
EXPECTED_ROWS = 15 * 3 * 15   # 15 subjek x 3 sesi x 15 trial = 675

print('=' * 60)
print('DATA QUALITY CONTROL AUDIT')
print('=' * 60)

for name, df in [('GC', df_gc), ('Combined', df_combined)]:
    print(f'\n--- {name} Dataset ---')
    print(f'  Shape          : {df.shape}')

    # Cek jumlah baris
    row_status = 'OK' if len(df) == EXPECTED_ROWS else f'WARNING! Expected {EXPECTED_ROWS}'
    print(f'  Row count      : {len(df)} ({row_status})')

    print(f'  Missing values : {df.isnull().sum().sum()}')
    print(f'  Duplicates     : {df.duplicated().sum()}')

    print(f'  Class distribution (class_label):')
    for cls, cnt in df['class_label'].value_counts().sort_index().items():
        balance = 'OK' if cnt == EXPECTED_ROWS // 3 else 'UNBALANCED!'
        print(f'    {cls:10s}: {cnt}  ({balance})')

    # Cek fitur konstan
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    const_cols = [c for c in numeric_cols if df[c].std() < 1e-10]
    if const_cols:
        print(f'  WARNING Constant features ({len(const_cols)}): {const_cols[:5]}')
    else:
        print(f'  Constant features : none')

    # Deteksi outlier dengan IQR
    outlier_cols = {}
    for col in numeric_cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        n_out = int(((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum())
        if n_out > 0:
            outlier_cols[col] = n_out

    if outlier_cols:
        top5 = sorted(outlier_cols.items(), key=lambda x: -x[1])[:5]
        print(f'  Outlier features  : {len(outlier_cols)}/{len(numeric_cols)} features affected')
        for col, cnt in top5:
            print(f'    {col}: {cnt} outliers')
    else:
        print(f'  Outlier features  : none')

print(f'\nQuality check complete.')
print(f'Output folder: {OUTPUT_DIR}')
print(f'  Files: {sorted(os.listdir(OUTPUT_DIR))}')


DATA QUALITY CONTROL AUDIT

--- GC Dataset ---
  Shape          : (675, 4047)
  Row count      : 675 (OK)
  Missing values : 0
  Duplicates     : 0
  Class distribution (class_label):
    negative  : 225  (OK)
    neutral   : 225  (OK)
    positive  : 225  (OK)
  Constant features : none
  Outlier features  : 4039/4043 features affected
    gc_deg_in_CPz: 152 outliers
    gc_str_in_CPz: 152 outliers
    gc_str_in_CP4: 108 outliers
    gc_edge_TP7_to_CPz: 107 outliers
    gc_edge_C4_to_PO3: 106 outliers

--- Combined Dataset ---
  Shape          : (675, 28293)
  Row count      : 675 (OK)
  Missing values : 0
  Duplicates     : 0
  Class distribution (class_label):
    negative  : 225  (OK)
    neutral   : 225  (OK)
    positive  : 225  (OK)
  Constant features : none
  Outlier features  : 27582/28283 features affected
    pdc_broadband_deg_out_C2: 167 outliers
    pdc_broadband_str_out_C2: 167 outliers
    pdc_beta_deg_out_C2: 166 outliers
    pdc_beta_str_out_C2: 166 outliers
    pdc_d